# Companion Lab 03 — Inspecting Next-Token Probabilities in a Real LLM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/5x5x5x5/taihls/blob/curriculum-v2/notebooks/03-clinical-llm-probabilities.ipynb)

In **Chapters 4.3–4.5**, we built toy next-token predictors and self-attention from scratch. Here we load a real pretrained Transformer (`distilgpt2`, ~82M parameters — fast on Colab CPU or GPU!) to **see its live top-10 next-token probability distribution**, test **temperature sweeps**, and watch **hallucination** happen in real time.

In [ ]:
!pip install -q transformers torch matplotlib

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'distilgpt2'   # Tip: try 'gpt2' or 'Qwen/Qwen2.5-0.5B' on a Colab GPU!
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device).eval()
print(f'Loaded {model_name} ({model.num_parameters():,} parameters) on {device}')

## 1. Peek directly inside the next-token probability distribution

In [ ]:
def plot_next_tokens(prompt, temperature=1.0, top_k=10):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1, :] / temperature
    probs = F.softmax(logits, dim=-1)
    top = torch.topk(probs, top_k)
    toks = [repr(tokenizer.decode(i)) for i in top.indices]
    vals = top.values.cpu().numpy() * 100

    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    ax.barh(toks[::-1], vals[::-1], color='#0072B2')
    ax.set_xlabel('Probability (%)')
    ax.set_title(f'Top {top_k} next tokens after {prompt!r} (T={temperature})')
    ax.grid(axis='x', alpha=0.25)
    plt.tight_layout()
    plt.show()

plot_next_tokens('The patient has a severe allergy to', temperature=1.0)

## 2. Temperature sweep & watching hallucinations happen

In [ ]:
def complete(prompt, temperature=0.8, max_new_tokens=35):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=True, temperature=temperature, top_k=50,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

for T in (0.3, 1.0, 1.8):
    print(f'--- Temperature T = {T} ---')
    print(complete('In a 2024 clinical trial published in The Lancet, the new drug', temperature=T))
    print()

## 3. Modify & Make (PRIMM challenges)

1. **Spot a fluent hallucination:** Look at the `In a 2024 clinical trial...` completions above (`distilgpt2` was trained in 2019 — it literally CANNOT know any 2024 study!). Notice how smoothly it fabricates numbers and medical claims anyway!
2. **Test grounding (Chapter 4.5):** Prefix the prompt with a short real fact (*"Fact: Aspirin is avoided in children with viral fever due to Reye syndrome. Question: Why should children with viral fever avoid aspirin? Answer:"*) and compare the top next-token probabilities before and after grounding!